# OpenPI pi0.5 xArm LoRA Fine-Tuning Pipeline

This notebook keeps most repeatable work on your local computer and Google Drive. Colab is used for OpenPI setup, final LeRobot conversion when needed, norm stats, and GPU training.

Persistent Drive layout:

```text
MyDrive/embodied_ai_xarm/
  xarm_colab_payload.zip          # uploaded from local computer
  lerobot/                        # persistent LeRobotDataset cache
  openpi_cache/pi05_base/params   # persistent pi05_base checkpoint
  openpi_assets/                  # persistent norm stats backup
  checkpoints/                    # persistent training outputs
```


## Local One-Time Prep

Run this on your Windows machine after code/data changes, then upload `xarm_colab_payload.zip` to `MyDrive/embodied_ai_xarm/`.

```powershell
Compress-Archive -Force -DestinationPath xarm_colab_payload.zip -Path `
  fine_tune\convert_xarm_raw_to_lerobot.py, `
  fine_tune\openpi_xarm_config.py, `
  fine_tune\data\xarm_pi05_data\raw
```


## 1. Mount Drive and Configure Paths

Run this at the start of every Colab session. It defines all persistent paths and points Hugging Face / uv / pip caches at Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/embodied_ai_xarm')
PAYLOAD_ZIP = DRIVE_ROOT / 'xarm_colab_payload.zip'
PROJECT_DIR = Path('/content/embodied-ai-xarm')
OPENPI_DIR = Path('/content/openpi')
HF_LEROBOT_HOME = DRIVE_ROOT / 'lerobot'
PI05_BASE_PARAMS = DRIVE_ROOT / 'openpi_cache/pi05_base/params'
ASSETS_BACKUP = DRIVE_ROOT / 'openpi_assets'
CHECKPOINT_BACKUP = DRIVE_ROOT / 'checkpoints'

# Google Drive is reliable for datasets/checkpoints, but not for uv/pip lock and temp files.
for p in [DRIVE_ROOT, HF_LEROBOT_HOME, PI05_BASE_PARAMS.parent, ASSETS_BACKUP, CHECKPOINT_BACKUP]:
    p.mkdir(parents=True, exist_ok=True)
Path('/content/uv_cache').mkdir(parents=True, exist_ok=True)
Path('/content/pip_cache').mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(HF_LEROBOT_HOME)
os.environ['UV_CACHE_DIR'] = '/content/uv_cache'
os.environ['PIP_CACHE_DIR'] = '/content/pip_cache'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

print('Drive root:', DRIVE_ROOT)
print('Payload exists:', PAYLOAD_ZIP.exists(), PAYLOAD_ZIP)
print('UV cache:', os.environ['UV_CACHE_DIR'])


## 2. Check GPU

A100 is preferred. L4 may work with small batch sizes. T4 is usually not enough for practical pi0.5 LoRA training.


In [ ]:
! nvidia-smi


## 3. Unpack Your Local Payload

Run this whenever you upload a new `xarm_colab_payload.zip` from your local computer. This is cheap and keeps Colab synchronized with local code/data.


In [ ]:
assert PAYLOAD_ZIP.exists(), f'Missing {PAYLOAD_ZIP}. Upload xarm_colab_payload.zip to Drive first.'
!rm -rf /content/embodied-ai-xarm
!mkdir -p /content/embodied-ai-xarm
!unzip -q "{PAYLOAD_ZIP}" -d /content/embodied-ai-xarm

# Compress-Archive may place individual files at the zip root. Normalize layout.
!mkdir -p /content/embodied-ai-xarm/fine_tune
!test -f /content/embodied-ai-xarm/openpi_xarm_config.py && cp /content/embodied-ai-xarm/openpi_xarm_config.py /content/embodied-ai-xarm/fine_tune/openpi_xarm_config.py || true
!test -f /content/embodied-ai-xarm/convert_xarm_raw_to_lerobot.py && cp /content/embodied-ai-xarm/convert_xarm_raw_to_lerobot.py /content/embodied-ai-xarm/fine_tune/convert_xarm_raw_to_lerobot.py || true

!find /content/embodied-ai-xarm -maxdepth 4 -type d | head -30
!ls /content/embodied-ai-xarm/fine_tune
!ls /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/raw


In [ ]:
# Create the missing directory structure
!mkdir -p /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data

# Move the raw folder into the correct path
!mv /content/embodied-ai-xarm/raw /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/

# Check to make sure it's there
!ls /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/raw

## 4. Clone and Install OpenPI

OpenPI lives in Colab `/content` for speed. Dependency caches live on Drive, so repeated installs should be faster.


In [ ]:
%%bash
set -e
cd /content
if [ ! -d openpi ]; then
  git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
fi
cd /content/openpi
pip install -q uv

# Temporarily move the cache to the local disk and disable hardlinks
export UV_CACHE_DIR=/tmp/uv_cache
export UV_LINK_MODE=copy

GIT_LFS_SKIP_SMUDGE=1 uv sync
GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

## 5. Patch OpenPI Config for xArm

This inserts `LeRobotXArmDataConfig` and the two xArm `TrainConfig`s. It also redirects the pi05 base weight loader to the Drive checkpoint cache.


In [ ]:
from pathlib import Path
import runpy

openpi_config = OPENPI_DIR / 'src/openpi/training/config.py'
snippet_file = PROJECT_DIR / 'fine_tune/openpi_xarm_config.py'
snippet = runpy.run_path(str(snippet_file))['XARM_CONFIG_SNIPPET']
marker = '# Add these TrainConfig entries to _CONFIGS.'
class_part, train_tail = snippet.split(marker, 1)
train_part = marker + train_tail

text = openpi_config.read_text()
if 'import openpi.policies.libero_policy as libero_policy' not in text:
    text = text.replace(
        'import openpi.policies.aloha_policy as aloha_policy\n',
        'import openpi.policies.aloha_policy as aloha_policy\nimport openpi.policies.libero_policy as libero_policy\n',
    )
if 'class LeRobotXArmDataConfig' not in text:
    text = text.replace('_CONFIGS = [', class_part.rstrip() + '\n\n_CONFIGS = [', 1)
if 'pi05_xarm_colab_smoke' not in text:
    text = text.replace('_CONFIGS = [', '_CONFIGS = [\n' + train_part.rstrip() + '\n', 1)

drive_weight_loader = f'weight_loader=weight_loaders.CheckpointWeightLoader("{PI05_BASE_PARAMS.as_posix()}")'
text = text.replace(
    'weight_loader=weight_loaders.CheckpointWeightLoader("gs://openpi-assets/checkpoints/pi05_base/params")',
    drive_weight_loader,
)
openpi_config.write_text(text)

!grep -n "LeRobotXArmDataConfig\|pi05_xarm_colab_smoke\|openpi_cache/pi05_base/params" /content/openpi/src/openpi/training/config.py


## 6. Convert Raw Data to Persistent LeRobotDataset

Run this only when raw data or the converter changes. The output is saved under Drive via `HF_LEROBOT_HOME`, so future sessions can skip this cell.


In [ ]:
%cd /content/openpi
!uv run python /content/embodied-ai-xarm/fine_tune/convert_xarm_raw_to_lerobot.py   --raw-root /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/raw   --output-dir /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/lerobot   --repo-id local/xarm_pi05_data   --robot-type xarm6   --fps 10   --overwrite
!find /content/drive/MyDrive/embodied_ai_xarm/lerobot -maxdepth 3 -type d | head -20


## 7. Cache pi05 Base Checkpoint on Drive

Run this only if `openpi_cache/pi05_base/params` is missing or incomplete. Once cached, training restores from Drive instead of repeatedly downloading from GCS.


In [ ]:
if not (PI05_BASE_PARAMS / 'commit_success.txt').exists():
    print('Caching pi05_base params to Drive. This may take a while.')
    !pip uninstall -y crcmod >/dev/null 2>&1 || true
    !pip install --no-cache-dir -U crcmod
    !python -c "import crcmod._crcfunext; print('compiled crcmod OK')" || printf "[GSUtil]\\ncheck_hashes = if_fast_else_skip\\n" > ~/.boto
    !rm -rf "{PI05_BASE_PARAMS}" "{PI05_BASE_PARAMS}.partial"
    !mkdir -p "{PI05_BASE_PARAMS}"
    !gsutil -m cp -r gs://openpi-assets/checkpoints/pi05_base/params/* "{PI05_BASE_PARAMS}/"
else:
    print('pi05_base params already cached:', PI05_BASE_PARAMS)
!ls -lah "{PI05_BASE_PARAMS}" | head


## 8. Compute or Restore Norm Stats

Run this when data/action definitions change. The stats are copied to Drive so a future session can restore them without recomputing.


In [ ]:
%%bash
set -e
cd /content/openpi

DATASET_INFO=/content/drive/MyDrive/embodied_ai_xarm/lerobot/local/xarm_pi05_data/meta/info.json
if [ ! -f "$DATASET_INFO" ]; then
  echo "Missing LeRobotDataset metadata: $DATASET_INFO"
  echo "Run Step 6 first and confirm it prints a LeRobot dataset path."
  find /content/drive/MyDrive/embodied_ai_xarm/lerobot -maxdepth 5 -name info.json || true
  exit 2
fi

if [ -d /content/drive/MyDrive/embodied_ai_xarm/openpi_assets/pi05_xarm_colab_smoke ]; then
  mkdir -p /content/openpi/assets
  cp -r /content/drive/MyDrive/embodied_ai_xarm/openpi_assets/pi05_xarm_colab_smoke /content/openpi/assets/
  echo "Restored norm stats from Drive."
else
  uv run scripts/compute_norm_stats.py --config-name pi05_xarm_colab_smoke
  mkdir -p /content/drive/MyDrive/embodied_ai_xarm/openpi_assets
  cp -r /content/openpi/assets/pi05_xarm_colab_smoke /content/drive/MyDrive/embodied_ai_xarm/openpi_assets/
  echo "Computed and backed up norm stats."
fi


## 9. Smoke Train

This runs the 1,000-step LoRA smoke test. Use it after pipeline changes before launching a longer run.


In [ ]:
%cd /content/openpi
!uv run scripts/train.py pi05_xarm_colab_smoke --exp-name=xarm_test --overwrite


## 10. Back Up Smoke Checkpoint

Run this immediately after training completes. Colab `/content` is temporary; Drive is persistent.


In [ ]:
!mkdir -p /content/drive/MyDrive/embodied_ai_xarm/checkpoints
!cp -r /content/openpi/checkpoints/pi05_xarm_colab_smoke /content/drive/MyDrive/embodied_ai_xarm/checkpoints/
!find /content/drive/MyDrive/embodied_ai_xarm/checkpoints/pi05_xarm_colab_smoke -maxdepth 3 -type d | sort | tail -20


## 11. Longer Training

Use this only after the smoke run finishes cleanly. For your current small dataset, consider editing `pi05_xarm` to 5,000 steps first, then scale up.


In [ ]:
%cd /content/openpi
# Optional: compute/restore pi05_xarm stats separately if you switch config names.
!uv run scripts/compute_norm_stats.py --config-name pi05_xarm
!uv run scripts/train.py pi05_xarm --exp-name=xarm_pi05_lora --overwrite
